In [18]:
from langchain_chroma import Chroma
from langchain_google_genai import GoogleGenerativeAIEmbeddings
from langchain_core.documents import Document
from langchain_huggingface import HuggingFaceEndpointEmbeddings

In [6]:
# Create LangChain documents for IPL players

doc1 = Document(
        page_content="Virat Kohli is one of the most successful and consistent batsmen in IPL history. Known for his aggressive batting style and fitness, he has led the Royal Challengers Bangalore in multiple seasons.",
        metadata={"team": "Royal Challengers Bangalore"}
    )
doc2 = Document(
        page_content="Rohit Sharma is the most successful captain in IPL history, leading Mumbai Indians to five titles. He's known for his calm demeanor and ability to play big innings under pressure.",
        metadata={"team": "Mumbai Indians"}
    )
doc3 = Document(
        page_content="MS Dhoni, famously known as Captain Cool, has led Chennai Super Kings to multiple IPL titles. His finishing skills, wicketkeeping, and leadership are legendary.",
        metadata={"team": "Chennai Super Kings"}
    )
doc4 = Document(
        page_content="Jasprit Bumrah is considered one of the best fast bowlers in T20 cricket. Playing for Mumbai Indians, he is known for his yorkers and death-over expertise.",
        metadata={"team": "Mumbai Indians"}
    )
doc5 = Document(
        page_content="Ravindra Jadeja is a dynamic all-rounder who contributes with both bat and ball. Representing Chennai Super Kings, his quick fielding and match-winning performances make him a key player.",
        metadata={"team": "Chennai Super Kings"}
    )


In [8]:
docs = [doc1, doc2, doc3, doc4, doc5]

In [19]:
# embeddings = GoogleGenerativeAIEmbeddings(
#     model="gemini-embedding-2",
#     task_type="SEMANTIC_SIMILARITY"
# )

embeddings = HuggingFaceEndpointEmbeddings(repo_id="sentence-transformers/all-MiniLM-L6-v2", task="feature-extraction")


In [20]:
vector_store = Chroma(
    embedding_function= embeddings,
    persist_directory="my_chroma_db",
    collection_name="samples"
)

In [21]:
vector_store.add_documents(docs)

['dc9a23f5-dfa3-44ac-b172-665e2811f30c',
 'fa8c76db-cb57-48ca-8914-0df6d618dc21',
 '4e910e18-2f99-4b3a-9016-684048daac8c',
 'e2405256-b293-4d4b-8d39-c5af80f8a9ab',
 '82b31654-5623-49e8-a9be-ce3d50e339ac']

In [23]:
vector_store.get(include=["metadatas", "documents", "embeddings"])

{'ids': ['dc9a23f5-dfa3-44ac-b172-665e2811f30c',
  'fa8c76db-cb57-48ca-8914-0df6d618dc21',
  '4e910e18-2f99-4b3a-9016-684048daac8c',
  'e2405256-b293-4d4b-8d39-c5af80f8a9ab',
  '82b31654-5623-49e8-a9be-ce3d50e339ac'],
 'embeddings': array([[ 0.00994718,  0.06914335, -0.05147116, ..., -0.03543343,
          0.0128481 ,  0.01248287],
        [ 0.0012775 ,  0.03129857, -0.0237538 , ..., -0.0051836 ,
         -0.03280616,  0.02737708],
        [-0.10265914,  0.02650807,  0.02271501, ..., -0.03359747,
         -0.07984945, -0.01507713],
        [ 0.0212339 , -0.02468549, -0.0449436 , ..., -0.10995808,
          0.00572554,  0.09915374],
        [ 0.01873968,  0.04382843, -0.04304254, ..., -0.07801618,
         -0.07840687, -0.00304189]]),
 'documents': ['Virat Kohli is one of the most successful and consistent batsmen in IPL history. Known for his aggressive batting style and fitness, he has led the Royal Challengers Bangalore in multiple seasons.',
  "Rohit Sharma is the most successful ca

In [44]:
query = "Who is the best captain?"

In [45]:
vector_store.similarity_search(
    query=query,
    k=2
)

[Document(id='fa8c76db-cb57-48ca-8914-0df6d618dc21', metadata={'team': 'Mumbai Indians'}, page_content="Rohit Sharma is the most successful captain in IPL history, leading Mumbai Indians to five titles. He's known for his calm demeanor and ability to play big innings under pressure."),
 Document(id='4e910e18-2f99-4b3a-9016-684048daac8c', metadata={'team': 'Chennai Super Kings'}, page_content='MS Dhoni, famously known as Captain Cool, has led Chennai Super Kings to multiple IPL titles. His finishing skills, wicketkeeping, and leadership are legendary.')]

In [46]:
vector_store.similarity_search_with_score(
    query=query,
    k=2
)

[(Document(id='fa8c76db-cb57-48ca-8914-0df6d618dc21', metadata={'team': 'Mumbai Indians'}, page_content="Rohit Sharma is the most successful captain in IPL history, leading Mumbai Indians to five titles. He's known for his calm demeanor and ability to play big innings under pressure."),
  0.8352534770965576),
 (Document(id='4e910e18-2f99-4b3a-9016-684048daac8c', metadata={'team': 'Chennai Super Kings'}, page_content='MS Dhoni, famously known as Captain Cool, has led Chennai Super Kings to multiple IPL titles. His finishing skills, wicketkeeping, and leadership are legendary.'),
  1.077178955078125)]

In [47]:
vector_store.similarity_search_with_relevance_scores(
    query=query,
    k=2,
    filter={'team': 'Chennai Super Kings'}
)


[(Document(id='4e910e18-2f99-4b3a-9016-684048daac8c', metadata={'team': 'Chennai Super Kings'}, page_content='MS Dhoni, famously known as Captain Cool, has led Chennai Super Kings to multiple IPL titles. His finishing skills, wicketkeeping, and leadership are legendary.'),
  0.23831945631281837),
 (Document(id='82b31654-5623-49e8-a9be-ce3d50e339ac', metadata={'team': 'Chennai Super Kings'}, page_content='Ravindra Jadeja is a dynamic all-rounder who contributes with both bat and ball. Representing Chennai Super Kings, his quick fielding and match-winning performances make him a key player.'),
  0.1198058900480653)]

In [48]:
# update documents
updated_doc1 = Document(
    page_content="Virat Kohli, the former captain of Royal Challengers Bangalore (RCB), is renowned for his aggressive leadership and consistent batting performances. He holds the record for the most runs in IPL history, including multiple centuries in a single season. Despite RCB not winning an IPL title under his captaincy, Kohli's passion and fitness set a benchmark for the league. His ability to chase targets and anchor innings has made him one of the most dependable players in T20 cricket.",
    metadata={"team": "Royal Challengers Bangalore"}
)


In [49]:
vector_store.update_document(document_id="dc9a23f5-dfa3-44ac-b172-665e2811f30c",document=updated_doc1)


In [50]:
vector_store.get(include=['embeddings','documents', 'metadatas'])

{'ids': ['dc9a23f5-dfa3-44ac-b172-665e2811f30c',
  'fa8c76db-cb57-48ca-8914-0df6d618dc21',
  '4e910e18-2f99-4b3a-9016-684048daac8c',
  'e2405256-b293-4d4b-8d39-c5af80f8a9ab',
  '82b31654-5623-49e8-a9be-ce3d50e339ac'],
 'embeddings': array([[-0.00233748,  0.05902077, -0.04774045, ..., -0.07264046,
          0.00276782, -0.00344092],
        [ 0.0012775 ,  0.03129857, -0.0237538 , ..., -0.0051836 ,
         -0.03280616,  0.02737708],
        [-0.10265914,  0.02650807,  0.02271501, ..., -0.03359747,
         -0.07984945, -0.01507713],
        [ 0.0212339 , -0.02468549, -0.0449436 , ..., -0.10995808,
          0.00572554,  0.09915374],
        [ 0.01873968,  0.04382843, -0.04304254, ..., -0.07801618,
         -0.07840687, -0.00304189]]),
 'documents': ["Virat Kohli, the former captain of Royal Challengers Bangalore (RCB), is renowned for his aggressive leadership and consistent batting performances. He holds the record for the most runs in IPL history, including multiple centuries in a sin

In [51]:
vector_store.delete('82b31654-5623-49e8-a9be-ce3d50e339ac')

In [52]:
vector_store.get(include=['embeddings','documents', 'metadatas'])

{'ids': ['dc9a23f5-dfa3-44ac-b172-665e2811f30c',
  'fa8c76db-cb57-48ca-8914-0df6d618dc21',
  '4e910e18-2f99-4b3a-9016-684048daac8c',
  'e2405256-b293-4d4b-8d39-c5af80f8a9ab'],
 'embeddings': array([[-0.00233748,  0.05902077, -0.04774045, ..., -0.07264046,
          0.00276782, -0.00344092],
        [ 0.0012775 ,  0.03129857, -0.0237538 , ..., -0.0051836 ,
         -0.03280616,  0.02737708],
        [-0.10265914,  0.02650807,  0.02271501, ..., -0.03359747,
         -0.07984945, -0.01507713],
        [ 0.0212339 , -0.02468549, -0.0449436 , ..., -0.10995808,
          0.00572554,  0.09915374]]),
 'documents': ["Virat Kohli, the former captain of Royal Challengers Bangalore (RCB), is renowned for his aggressive leadership and consistent batting performances. He holds the record for the most runs in IPL history, including multiple centuries in a single season. Despite RCB not winning an IPL title under his captaincy, Kohli's passion and fitness set a benchmark for the league. His ability to